# 04b: Extending the bound to keV screening and closer D–D spacings

Two inputs found after notebook 04:

* **Karahadian et al. (Nat. Commun. 2026, arXiv:2512.06212, read in full):** fitted screening potentials of **0.8±0.2 keV (Pd, beam-loaded), 1.7±0.2 keV (PdD, beam + electrochemical loading), 0.3±0.2 keV (Ti), 0.9±0.2 keV (TiD)**, plus a yield plateau below 2.5 keV. The authors note that U_e may lump several mechanisms (depth-dependent loading, near-surface deuterium replenishment, possible resonances).
* **Sun & Tománek (1989, via reviews):** deuterons in heavily loaded Pd can sit in equilibrium as close as **0.94 Å** (average ~1.4 Å), versus the 2.75 Å used in notebooks 03–04.

Same analysis as notebook 04: lattice pair, profile matched to U_meas at the beam turning point (0.004 Å), effective rate including the short-range bound state. The scan starts at ℓ = 0.002 Å so that keV cases are resolved (below that, matching the beam data needs contact strengths of hundreds of keV).

In [1]:
import numpy as np, matplotlib.pyplot as plt
from cfsolve import *
NU0 = 37e-3/4.135667e-15; LL = np.log10(2e-26); LW = np.log10(5.6e-11)
ELLS = np.logspace(np.log10(0.002), np.log10(0.4), 64)
def band(d0A, U, shape):
    Vb = V_lattice(d0_A=d0A); d0 = d0A/BOHR_A; Rmax = d0 + 7; Rmin = 0.4*d0
    E_lat = solve(Vb, Rmax, Rmin_well=Rmin)["E"]; out = []
    for l in ELLS:
        ex = extra_screening(beam_matched_dU(U, l, shape), l, shape); V = lambda R: Vb(R) + ex(R)
        try: out.append(solve(V, Rmax, Rmin_well=Rmin)["log10_rate"])
        except Collapse:
            B = solve(V, Rmax, Rmin_well=0.0); r = np.logspace(-4, np.log10(Rmax), 200000); v = V(r)
            imin = np.argmin(np.where(r < Rmin, v, np.inf)); sel = (r > r[imin]) & (r < d0) & (v > E_lat)
            G = 2*np.trapezoid(np.sqrt(2*MU_DD*np.clip(v[sel]-E_lat, 0, None)), r[sel])
            out.append(min(np.log10(NU0) - G/np.log(10), B["log10_rate"]))
    return np.array(out)
for d in [2.75, 1.4, 0.94]:
    print(f"bare lattice pair at {d} A: 10^{solve(V_lattice(d0_A=d), d/BOHR_A + 7, Rmin_well=0.4*d/BOHR_A)['log10_rate']:.1f} /s")

bare lattice pair at 2.75 A: 10^-123.9 /s


bare lattice pair at 1.4 A: 10^-112.1 /s


bare lattice pair at 0.94 A: 10^-109.5 /s


In [2]:
rows = []
for U in [300, 575, 800, 900, 1700, 2000]:
    for d in [2.75, 1.4, 0.94]:
        for sh in ["exponential", "gaussian", "step", "yukawa"]:
            e = band(d, U, sh)
            f = lambda t: ELLS[np.where(e > t)[0][0]] if np.any(e > t) else np.nan
            n, w = f(LL), f(LW)
            k = np.where(e > LL)[0]
            leap = bool(len(k) and k[0] > 0 and e[k[0]] > LW - 2)       # jump from below the null limit to near-watt in one step
            rows.append((U, d, sh, n, w, e.max(), leap))
print(f"{'U_meas':>6s} {'d (A)':>6s} {'shape':12s} {'null at':>8s} {'watts at':>9s} {'max rate':>9s}  watts-before-neutrons?")
for U, d, sh, n, w, m, leap in rows:
    fm = lambda x: "   —   " if not np.isfinite(x) else f"{x:7.4f}"
    print(f"{U:6d} {d:6.2f} {sh:12s} {fm(n):>8s} {fm(w):>9s} {m:9.1f}  {'YES' if leap or (np.isfinite(w) and (not np.isfinite(n) or w < n)) else 'no'}")

U_meas  d (A) shape         null at  watts at  max rate  watts-before-neutrons?
   300   2.75 exponential    0.1586      —        -18.0  no
   300   2.75 gaussian       0.3108      —        -22.9  no
   300   2.75 step             —         —        -36.3  no
   300   2.75 yukawa         0.0413      —        -19.7  no
   300   1.40 exponential    0.1232      —        -17.1  no
   300   1.40 gaussian       0.2220      —        -20.0  no
   300   1.40 step             —         —        -29.6  no
   300   1.40 yukawa         0.0413      —        -19.6  no
   300   0.94 exponential    0.1133      —        -16.5  no
   300   0.94 gaussian       0.1725      —        -17.3  no
   300   0.94 step             —         —        -26.2  no
   300   0.94 yukawa         0.0413      —        -19.6  no
   575   2.75 exponential    0.1340      —        -11.3  no
   575   2.75 gaussian       0.3108      —        -18.6  no
   575   2.75 step             —         —        -34.4  no
   575   2.75 yukawa

In [3]:
arr = [(U, d, sh, n, w) for U, d, sh, n, w, m, leap in rows]
viol = [r for r in rows if r[6] or (np.isfinite(r[4]) and (not np.isfinite(r[3]) or r[4] < r[3]))]
ratios = [w/n for U, d, sh, n, w in arr if np.isfinite(w) and np.isfinite(n)]
nulls = [n for U, d, sh, n, w in arr if np.isfinite(n)]
print(f"variants: {len(rows)};  reach watts: {sum(np.isfinite(r[4]) for r in rows)};  watts-before-neutrons: {len(viol)}")
print(f"smallest ratio (range at watts / range at null limit): {min(ratios):.2f}")
print(f"range bound across constrained variants: {min(nulls):.4f} – {max(nulls):.3f} A")
for U in [900, 1700, 2000]:
    y = [n for (UU, d, sh, n, w) in arr if UU == U and sh == "yukawa"]
    ex = [n for (UU, d, sh, n, w) in arr if UU == U and sh == "exponential"]
    print(f"U_meas = {U} eV: Yukawa bound {min(y):.4f}–{max(y):.4f} A; exponential {min(ex):.3f}–{max(ex):.3f} A")

variants: 72;  reach watts: 33;  watts-before-neutrons: 0
smallest ratio (range at watts / range at null limit): 1.29
range bound across constrained variants: 0.0030 – 0.311 A
U_meas = 900 eV: Yukawa bound 0.0108–0.0108 A; exponential 0.074–0.123 A
U_meas = 1700 eV: Yukawa bound 0.0043–0.0043 A; exponential 0.063–0.113 A
U_meas = 2000 eV: Yukawa bound 0.0030–0.0030 A; exponential 0.063–0.104 A


## Result

* **"Watts imply neutrons" holds for every variant**, including the keV screening values of Karahadian et al. and the closer equilibrium spacings. 33 of the 72 variants reach watt level somewhere; all cross the neutron limit first. The tightest case (Yukawa profile at 1.7 keV) has a margin of only 1.29 in range.
* **For keV-scale screening the range bound collapses toward the beam turning point:** with a long-tailed (Yukawa) profile, ≈0.003–0.004 Å. In other words, keV screening cannot extend appreciably beyond the distances the beam itself probes without violating the 1990 null limit.
* Closer equilibrium spacings (0.94–1.4 Å) raise the bare rate by up to 14 orders but tighten, rather than loosen, the bound.